In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent.parent))

from data.OT2.ot2 import PUMP
from otp.core.units import BAR, K_TO_C, P_A, g
from otp.propellants.propellant import Propellant
from pyfluids import FluidsList, Input
from otp.experiments.loaders.airborne import load_run
from scipy.optimize import brentq
import numpy as np
import matplotlib.pyplot as plt

T_K = 293.15
P_IN = 10 * BAR

def rho(fl):
    return fl.fluid.with_state(Input.pressure(1 * BAR), Input.temperature(T_K + K_TO_C)).density

WATER = Propellant("Water", FluidsList.Water, t_tank=T_K, p_tank=P_IN)
class IPA:                      # CoolProp has no propan-2-ol; 20 C literature values
    class _s: density, kinematic_viscosity, pressure = 785.4, 2.66e-6, 4400.0
    class fluid:
        with_state = staticmethod(lambda *a: IPA._s)   # .pressure only read for p_sat

from data.OT2.ot2 import CHANNEL_MAP, DATA_PATH
run = load_run(DATA_PATH, cmap=CHANNEL_MAP)
STEADY_STATE_START, STEADY_STATE_END = (5.0, 10.0)

p_orifice_in = run["p_orifice_in"].mean(STEADY_STATE_START, STEADY_STATE_END)
print(f"p_orifice_in = {p_orifice_in:.2f} bar")

# Tests
IPA_MDOT = np.array([0.706, 0.503, 0.301]) #kg/s
IPA_DP = np.array([12.84, 6.59, 2.31]) * BAR #Pa

CdA = IPA_MDOT / np.sqrt(2 * rho(IPA) * IPA_DP)

# Water Mdot
WATER_MDOT = CdA * np.sqrt(2 * rho(WATER) * (p_orifice_in - 1) * BAR)
print(CdA)
print(WATER_MDOT)
print("")
# Print mean of both
print(f"Mean CdA = {CdA.mean():.6g} m^2, Mean WATER_MDOT = {WATER_MDOT.mean():.3f} kg/s")


print(PUMP.point_performance(IPA, 1.7/787, 33000, 10*BAR, T_K, "lock"))

USER_HOME_DIR=C:\Users\Martin
p_orifice_in = 88.45 bar
[1.57203331e-05 1.56338101e-05 1.58015634e-05]
[2.07712033 2.06568809 2.08785326]

Mean CdA = 1.57186e-05 m^2, Mean WATER_MDOT = 2.077 kg/s
PumpPerformance
  ----- Inputs -----
    propellant                 <class '__main__.IPA'> pumped propellant               
    Q                                         0.00216 m^3/s                 2.16 L/s  pump flow rate
    rpm                                       3.3e+04 RPM                             pump rotational speed
    p_upstream                                  1e+06 Pa                      10 bar  pump upstream pressure
    T_upstream                                    293 K                       20 C    pump upstream temperature
  ----- Outputs -----
    omega                                    3.46e+03 rad/s                           pump rotational speed in rad/s
    mdot                                          1.7 kg/s                            pump mass flow rate
    dp